# Chapter 7 - Unsupervised Learning

## Chapter summary
**Unsupervised learning** extracts structure from data *without* a labelled outcome. This chapter covers:

1. **Principal Components Analysis (PCA)** - reduces correlated numeric variables to a few uncorrelated components.
2. **K-means clustering** - partitions records into $K$ groups around cluster means.
3. **Hierarchical clustering** - builds a tree (dendrogram) of nested clusters; linkage methods.
4. **Model-based clustering** - Gaussian mixture models fitted with the EM algorithm; choosing the number of clusters with BIC.
5. **Scaling and categorical variables** - standardisation, Gower's distance, treating mixed data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"

def get_data(name):
    """Download a dataset of the book (once) into ./data and return its path."""
    path = Path("data") / name
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + name, path)
    return path

sns.set_style("whitegrid")

sp500_px = pd.read_csv(get_data("sp500_data.csv.gz"), index_col=0)
sp500_sym = pd.read_csv(get_data("sp500_sectors.csv"))

## 1. Principal Components Analysis (PCA)

**Theory.** PCA finds new variables - **principal components** - that are *linear combinations* of the original variables, are mutually **uncorrelated**, and are ordered by the amount of variance they capture. The first component is the direction of greatest variance in the data; the second is orthogonal to it, and so on. It is computed from the eigenvectors of the covariance (or correlation) matrix.

* **Loadings** (weights): how each original variable contributes to a component.
* **Scree plot**: variance (or % variance) explained by each component; the "elbow" suggests how many to keep.
* **Standardise** variables first if they are on different scales (or use the correlation matrix).

Example: daily returns of Chevron and ExxonMobil (two highly correlated stocks).

In [ ]:
oil_px = sp500_px[["XOM", "CVX"]]
pca = PCA(n_components=2).fit(oil_px)
loadings = pd.DataFrame(pca.components_, columns=oil_px.columns, index=["PC1", "PC2"])
print(loadings)

In [ ]:
def abline(slope, intercept, ax):
    x_vals = np.array(ax.get_xlim())
    ax.plot(x_vals, intercept + slope * x_vals, "--", color="C1")

ax = oil_px.plot.scatter(x="XOM", y="CVX", alpha=0.3, figsize=(5, 5))
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
abline(loadings.loc["PC2", "CVX"] / loadings.loc["PC2", "XOM"], 0, ax)
abline(loadings.loc["PC1", "CVX"] / loadings.loc["PC1", "XOM"], 0, ax)
plt.title("Principal component directions"); plt.show()

**Interpreting the components** on a larger set: daily returns since 2011 of 25 well-known S&P 500 stocks from several sectors (tech, energy, financials, telecom, consumer).

In [ ]:
syms = sorted(["AAPL", "MSFT", "CSCO", "INTC", "GOOGL", "AMZN", "XOM", "CVX", "COP", "SLB", "OXY",
               "JPM", "WFC", "C", "BAC", "GS", "T", "VZ", "KO", "PEP", "PG", "WMT", "COST", "HD", "MCD"])
syms = [s for s in syms if s in sp500_px.columns]
top_sp = sp500_px.loc[sp500_px.index >= "2011-01-01", syms]

sp_pca = PCA().fit(top_sp)
explained = pd.DataFrame(sp_pca.explained_variance_ratio_, columns=["Explained variance"],
                         index=[f"PC{i + 1}" for i in range(len(syms))])
ax = explained.head(10).plot.bar(figsize=(6, 3.5), legend=False)
ax.set_ylabel("Explained variance ratio"); plt.title("Scree plot"); plt.show()

loadings = pd.DataFrame(sp_pca.components_[:5, :], columns=top_sp.columns, index=[f"PC{i}" for i in range(1, 6)])
fig, axes = plt.subplots(5, 1, figsize=(9, 9), sharex=True)
for i, ax in enumerate(axes):
    pc = loadings.iloc[i, :]
    ax.bar(pc.index, pc.values, color=["C0" if v > 0 else "C3" for v in pc.values])
    ax.set_ylabel(f"PC{i + 1}")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

The first component loads on all stocks with the same sign (the overall market); later components contrast sectors (e.g. energy vs. tech).

## 2. K-Means Clustering

**Theory.** K-means divides the data into $K$ clusters by minimising the **within-cluster sum of squares** (sum of squared distances of records to their cluster mean):

1. Choose $K$ and initial cluster means.
2. Assign every record to the nearest mean.
3. Recompute each mean as the average of its records.
4. Repeat 2-3 until assignments stop changing.

Results depend on the starting means, so run with several **random starts** (`n_init`; `k-means++` helps). **Standardise** variables so that none dominates the distance. To choose $K$ use the **elbow method** (plot within-cluster SS vs. $K$ and look for the bend), or the silhouette score. *Cluster size, mean, and spread* describe each cluster.

Example: clustering XOM and CVX returns.

In [ ]:
df = sp500_px.loc[sp500_px.index >= "2011-01-01", ["XOM", "CVX"]]
kmeans = KMeans(n_clusters=4, n_init=10, random_state=1).fit(df)
df["cluster"] = kmeans.labels_
print("Cluster sizes:", df.cluster.value_counts().to_dict())
print(pd.DataFrame(kmeans.cluster_centers_, columns=["XOM", "CVX"]).round(3))

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x="XOM", y="CVX", hue="cluster", data=df, palette="tab10", s=12, ax=ax)
ax.plot(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], "k*", markersize=15)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); plt.show()

**Interpreting clusters** in a bigger problem (stock returns of the 25 chosen tickers from 2011) and choosing $K$ with the elbow method:

In [ ]:
X = top_sp.copy()
inertia = []
for k in range(2, 15):
    inertia.append(KMeans(n_clusters=k, n_init=10, random_state=1).fit(X).inertia_)
plt.figure(figsize=(5, 3.5)); plt.plot(range(2, 15), inertia, "o-")
plt.xlabel("Number of clusters K"); plt.ylabel("Within-cluster sum of squares"); plt.title("Elbow method"); plt.show()

# cluster centres of the 5-cluster solution
km5 = KMeans(n_clusters=5, n_init=10, random_state=1).fit(X)
centers = pd.DataFrame(km5.cluster_centers_, columns=X.columns)
print("Cluster sizes:", np.bincount(km5.labels_))
fig, axes = plt.subplots(5, 1, figsize=(9, 8), sharex=True)
for i, ax in enumerate(axes):
    ax.bar(centers.columns, centers.iloc[i], color=["C0" if v > 0 else "C3" for v in centers.iloc[i]])
    ax.set_ylabel(f"Cl {i}")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 3. Hierarchical Clustering

**Theory.** Hierarchical (agglomerative) clustering needs no prespecified $K$ and produces a **dendrogram**:
1. Start with every record as its own cluster.
2. Repeatedly merge the two *closest* clusters until one remains.

The notion of closeness between clusters is the **linkage**: **single** (minimum pairwise distance - can chain), **complete** (maximum - compact clusters), **average**, and **Ward** (merge that least increases total within-cluster variance, similar in spirit to K-means). Cutting the dendrogram at a height gives a given number of clusters. It is intuitive and works with any distance metric but is costly for large datasets.

In [ ]:
syms1 = [s for s in ["AAPL", "AMZN", "MSFT", "CSCO", "INTC", "XOM", "CVX", "COP", "SLB", "OXY", "JPM", "WFC", "C", "BAC", "GS",
                      "KO", "PEP", "PG", "WMT", "T", "VZ"] if s in sp500_px.columns]
df1 = sp500_px.loc[sp500_px.index >= "2011-01-01", syms1].T

Z = linkage(df1, method="complete")
fig, ax = plt.subplots(figsize=(8, 4))
dendrogram(Z, labels=list(df1.index), color_threshold=0, ax=ax)
ax.set_ylabel("Distance"); plt.title("Dendrogram (complete linkage)"); plt.xticks(rotation=90); plt.tight_layout(); plt.show()

memb = pd.Series(fcluster(Z, 4, criterion="maxclust"), index=df1.index)
for c in sorted(memb.unique()):
    print(f"Cluster {c}:", ", ".join(memb[memb == c].index))

In [ ]:
# Comparing linkage methods
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, method in zip(axes, ("single", "average", "ward")):
    dendrogram(linkage(df1, method=method), labels=list(df1.index), color_threshold=0, ax=ax)
    ax.set_title(method); ax.tick_params(axis="x", labelrotation=90)
plt.tight_layout(); plt.show()

## 4. Model-Based Clustering

**Theory.** Instead of purely distance-based rules, model-based clustering assumes the data come from a **mixture of multivariate normal distributions**, each with its own mean vector $\mu_k$ and covariance matrix $\Sigma_k$, and mixing proportion $\pi_k$:

$$ f(x)=\sum_{k=1}^{K}\pi_k\,\mathcal{N}(x\mid\mu_k,\Sigma_k). $$

Parameters are estimated with the **EM (expectation-maximisation) algorithm**: the E-step computes each record's probability ("responsibility") of belonging to each cluster; the M-step updates $\pi_k,\mu_k,\Sigma_k$. Clusters can have different shapes/sizes/orientations (controlled by `covariance_type`), and records get **soft** assignments. The number of clusters is chosen with the **Bayesian Information Criterion (BIC)** - lower is better in scikit-learn - which penalises extra parameters.

In [ ]:
df = sp500_px.loc[sp500_px.index >= "2011-01-01", ["XOM", "CVX"]]
mclust = GaussianMixture(n_components=2, covariance_type="full", n_init=5, random_state=1).fit(df)
print("Means:\n", pd.DataFrame(mclust.means_, columns=df.columns).round(3))
print("Weights:", mclust.weights_.round(3))
print("Covariance matrices:\n", mclust.covariances_.round(3))

fig, ax = plt.subplots(figsize=(5, 5))
labels = mclust.predict(df)
sns.scatterplot(x=df["XOM"], y=df["CVX"], hue=labels, palette="tab10", s=12, ax=ax)
ax.plot(mclust.means_[:, 0], mclust.means_[:, 1], "k*", markersize=15)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); plt.title("Gaussian mixture (2 components)"); plt.show()

In [ ]:
# Select the number of clusters and covariance structure with BIC
results = []
for cov in ("spherical", "diag", "tied", "full"):
    for k in range(1, 9):
        gm = GaussianMixture(n_components=k, covariance_type=cov, n_init=3, random_state=1).fit(df)
        results.append({"covariance": cov, "k": k, "BIC": gm.bic(df)})
results = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(6, 4))
for cov, g in results.groupby("covariance"):
    ax.plot(g.k, g.BIC, "o-", label=cov)
ax.set_xlabel("Number of components"); ax.set_ylabel("BIC (lower is better)"); ax.legend(); plt.show()
print(results.loc[results.BIC.idxmin()])

## 5. Scaling and Categorical Variables

**Theory.**
* **Scaling**: distance-based methods (K-means, hierarchical, KNN) depend on variable units. **Standardise** (z-score) so each numeric variable has mean 0 and SD 1. PCA on unscaled data is dominated by the variable with the largest variance. Scaling choices also let you *up-weight* a variable deliberately.
* **Dominant variables**: even after scaling, correlated groups of variables can dominate; PCA loadings reveal this.
* **Categorical data**: convert to dummy (one-hot) variables, which can over-weight them; alternatively use **Gower's distance**, which uses a different dissimilarity per variable type (range-scaled absolute difference for numeric, 0/1 mismatch for categorical) and averages them. It lets you apply hierarchical clustering to **mixed data**.
* **Problems with clustering mixed data**: K-means works only with numeric variables; for mixed data use Gower + hierarchical clustering (or K-prototypes).

In [ ]:
# PCA on unscaled vs. scaled loan data
loan_data = pd.read_csv(get_data("loan_data.csv.gz"))
cols = ["loan_amnt", "annual_inc", "revol_bal", "open_acc", "dti", "revol_util"]
loan_num = loan_data[cols].dropna().sample(5000, random_state=1)

pca_raw = PCA(n_components=2).fit(loan_num)
pca_std = PCA(n_components=2).fit(StandardScaler().fit_transform(loan_num))
print("Unscaled PC1 loadings:\n", pd.Series(pca_raw.components_[0], index=cols).round(3))
print("Standardised PC1 loadings:\n", pd.Series(pca_std.components_[0], index=cols).round(3))
print("Explained variance (unscaled):", pca_raw.explained_variance_ratio_.round(3))

In [ ]:
def gower_distance(df, num_cols, cat_cols):
    """Gower dissimilarity matrix for a mixed-type DataFrame (equal weights)."""
    n = len(df)
    D = np.zeros((n, n))
    for c in num_cols:
        v = df[c].values.astype(float)
        rng_ = v.max() - v.min() or 1.0
        D += np.abs(v[:, None] - v[None, :]) / rng_
    for c in cat_cols:
        v = df[c].values
        D += (v[:, None] != v[None, :]).astype(float)
    return D / (len(num_cols) + len(cat_cols))

mixed = loan_data[["dti", "payment_inc_ratio", "home_", "purpose_"]].dropna().sample(300, random_state=1).reset_index(drop=True)
D = gower_distance(mixed, ["dti", "payment_inc_ratio"], ["home_", "purpose_"])
Zg = linkage(squareform(D, checks=False), method="complete")
mixed["cluster"] = fcluster(Zg, 4, criterion="maxclust")
print(mixed.groupby("cluster").agg({"dti": "mean", "payment_inc_ratio": "mean", "home_": lambda s: s.mode()[0],
                                    "purpose_": lambda s: s.mode()[0], "cluster": "size"}).rename(columns={"cluster": "n"}))

## Key takeaways
* **PCA** compresses correlated variables into a few uncorrelated components; the scree plot shows how many to keep and loadings explain them.
* **K-means** is fast and simple; standardise the data, use multiple starts and the elbow method to choose $K$.
* **Hierarchical clustering** gives a dendrogram and works with any distance; linkage choice changes the result.
* **Model-based (GMM) clustering** gives soft assignments and a statistical way (BIC) to choose the number of clusters.
* Always **scale** variables; for mixed numeric/categorical data use **Gower's distance**.